In [86]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# Configure Polars to print everything
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns


polars.config.Config

In [2]:
KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

### Load data

In [3]:
CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SAMPLE_SUBMISSION = pl.read_csv(f"{INPUT_BASE_PATH}/sample_submission.csv")

In [12]:
full_train_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales",
    )
    .join(
        # Initially only joining on a subset of columns
        other=CALENDAR_DATA.select(["date", "wm_yr_wk", "d"]),
        on="d",
        how="left",
    )
    .join(
        other=SELL_PRICES,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
    # Remove columns where sell price is null
    .filter(pl.col("sell_price").is_not_null())
    # Add integer index
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int64))
)

In [13]:
# Check no null prices
assert not full_train_df["sell_price"].is_null().any()

In [11]:
# Check no duplicates
assert full_train_df.select(pl.col("id"), pl.col("d_index")).is_duplicated().sum() == 0

In [23]:
# Check continuous timeseries
id_series_lengths = (
    full_train_df
    .group_by("id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .with_columns(min_max_diff=pl.col("d_index_max") - pl.col("d_index_min"))
    .join(
        other=full_train_df.group_by("id").len(name="series_length"),
        on="id",
        how="inner",
        validate="1:1"
    )
)

assert id_series_lengths["min_max_diff"].eq(id_series_lengths["series_length"] - 1).all()

### Lagged features

In [ ]:
SALES_LAGS = [1, 2, 3, 4, 7, 14]

SALES_LAG_DF = (
    full_train_df
    .select(["id", "d_index", "sales"])
    .with_columns(
        [
            pl.col("sales").shift(lag).over("id", order_by="d_index").alias(f"sales_lagged_{lag}")
            for lag in SALES_LAGS
        ]
    )
    .sort(by=["id", "d_index"])
)

SALES_LAG_DF.head()

### Rolling features

In [ ]:
ROLLING_WINDOWS = [2, 3, 4, 7, 14]

item_ids = full_train_df["id"].unique().sort().to_list()[:2]
ROLLING_MEAN_DF = (
    full_train_df
    .select(["id", "d_index", "sales"])
    .with_columns(
        [
            pl.col("sales")
            .shift(1)
            .rolling_mean(window_size=window, min_samples=1)
            .over("id", order_by="d_index")
            .alias(f"sales_rolling_mean_{window}")
            for window in ROLLING_WINDOWS
        ]
    )
)

ROLLING_MEAN_DF.head()

id,d_index,sales,sales_rolling_mean_2,sales_rolling_mean_3,sales_rolling_mean_4,sales_rolling_mean_7,sales_rolling_mean_14
str,i64,i64,f64,f64,f64,f64,f64
"""HOBBIES_1_008_CA_1_evaluation""",1,12,null,null,null,null,null
"""HOBBIES_1_009_CA_1_evaluation""",1,2,null,null,null,null,null
"""HOBBIES_1_010_CA_1_evaluation""",1,0,null,null,null,null,null
"""HOBBIES_1_012_CA_1_evaluation""",1,0,null,null,null,null,null
"""HOBBIES_1_015_CA_1_evaluation""",1,4,null,null,null,null,null


### Lagged price features

#### TODO

### Rolling price features

#### TODO

### Event features

In [113]:
N_EVENTS = 2

EVENT_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(
        pl.col("d").str.strip_prefix("d_").cast(pl.Int64).alias("d_index")
    )
    .with_columns(
        [
            pl.when(pl.col(f"event_name_{i}").is_not_null())
            .then(pl.col("d_index"))
            .otherwise(None)
            .alias(f"event_index_{i}")
            for i in (1, 2)
        ]
    )
    # Days until next / since last event
    .with_columns(
        [
            # Days since last event
            *[
                (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
                .alias(f"days_since_event_{i}")
                for i in (1, 2)
            ],
            # Last event type
            *[
                (pl.col(f"event_type_{i}").forward_fill()
                .alias(f"last_event_type_{i}"))
                for i in range(1, N_EVENTS + 1)
            ],
            # Days until next event
            *[
                (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
                .alias(f"days_until_event_{i}")
                for i in (1, 2)
            ],
            # Next event type
            *[
                (pl.col(f"event_type_{i}").backward_fill()
                .alias(f"next_event_type_{i}"))
                for i in range(1, N_EVENTS + 1)
            ],
            
        ],
    )
    .with_columns(
        [
            # Clip days since last event and event type to 7 days.
            *[
                pl.when(pl.col(f"days_since_event_{i}") <= 7)
                .then(pl.col(f"days_since_event_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            *[
                pl.when(pl.col(f"days_since_event_{i}") <= 7).
                then(pl.col(f"last_event_type_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            # Clip days until next event and event type to 7 days.
            *[              
                pl.when(pl.col(f"days_until_event_{i}") <= 7)
                .then(pl.col(f"days_until_event_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ],
            *[
                pl.when(pl.col(f"days_until_event_{i}") <= 7)
                .then(pl.col(f"next_event_type_{i}"))
                .otherwise(None)
                for i in range(1, N_EVENTS + 1)
            ]
        ]
    )
)

EVENT_FEATURES_DF.head()

wm_yr_wk,d,event_name_1,event_type_1,event_name_2,event_type_2,d_index,event_index_1,event_index_2,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2
i64,str,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,i64,str,str
11101,"""d_1""",null,null,null,null,1,null,null,null,null,null,null,null,null,null,null
11101,"""d_2""",null,null,null,null,2,null,null,null,null,null,null,7,null,"""Sporting""",null
11101,"""d_3""",null,null,null,null,3,null,null,null,null,null,null,6,null,"""Sporting""",null
11101,"""d_4""",null,null,null,null,4,null,null,null,null,null,null,5,null,"""Sporting""",null
11101,"""d_5""",null,null,null,null,5,null,null,null,null,null,null,4,null,"""Sporting""",null


### Calendar features

In [58]:
CALENDAR_FEATURES_DF = (
    CALENDAR_DATA
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float16),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float16),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float16),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float16),
        is_weekend=(pl.col("wday") >= 6).cast(pl.Float16),
    )
)

CALENDAR_FEATURES_DF.head()

wm_yr_wk,d,date,wday,month,year,sin_wday,cos_wday,sin_month,cos_month,is_weekend
i64,str,date,i64,i64,i64,f16,f16,f16,f16,f16
11101,"""d_1""",2011-01-29,1,1,2011,0.781738,0.623535,0.5,0.866211,0.0
11101,"""d_2""",2011-01-30,2,1,2011,0.975098,-0.222534,0.5,0.866211,0.0
11101,"""d_3""",2011-01-31,3,1,2011,0.433838,-0.900879,0.5,0.866211,0.0
11101,"""d_4""",2011-02-01,4,2,2011,-0.433838,-0.900879,0.866211,0.5,0.0
11101,"""d_5""",2011-02-02,5,2,2011,-0.975098,-0.222534,0.866211,0.5,0.0


## MERGE ALL TRAIN FEATURES

## MERGE ALL SUBMISSION FEATURES

###